# Итоговая практическая работа по Теме 1 (ПР №4 и ПР №5)
## «Какой знак видит камера?»

**Цель:** пройти полный цикл решения задачи глубокого обучения так, как это делают на практике: определить метрику, найти и исправить проблемы в данных, честно разделить выборку, обучить и отладить базовую сверточную сеть, поставить диагноз, улучшить модель приемами Блока 2 в пределах ограниченного бюджета опытов, один раз проверить итоговую модель на тестовой выборке и защитить результат.

**Время:** 2 академических часа на работу, защита – на следующем занятии или в конце этого.

| Часть | Шаги | Что получится |
|---|---|---|
| ПР №4 «Данные, метрика и базовая модель» | 1–5 | чистые данные, честное разбиение, проверенная базовая сеть и диагноз |
| ПР №5 «Улучшение модели и итоговая проверка» | 6–10 | журнал опытов, лучшая модель, итог на тесте, отчет |

**Вариант.** У каждого свой файл данных `data/road_signs_vNN.npz`, где NN – номер варианта, который назовет учитель. Укажите его в первой готовой ячейке (`VARIANT = ...`). Числа у всех получатся разные, а ход работы одинаковый.

**Что можно.** Все, что было в Блоках 1 и 2: PyTorch, NumPy, pandas и метрики из scikit-learn. Чтобы два часа ушли на решения, а не на повторный набор кода, цикл обучения, сеть и рисунки уже написаны в готовых ячейках (шпаргалка ниже). Код каждого шага пишете вы.

**Правила.**

1. Везде, где есть случайность, используйте `SEED = 42` (функция `set_seed`), чтобы результат можно было воспроизвести.
2. Тестовая выборка `X_test` до шага 9 **не используется ни в каком виде**: ни для подбора, ни для рисунков, ни для «просто посмотреть».
3. Каждое обучение записывается в журнал опытов функцией `log`. В ПР №5 бюджет – **не больше 8 обучений** (шаги 6–8). Опыт, которого нет в журнале, на защите не считается.
4. После каждого шага есть вопросы. Ответ пишите своими словами и своими числами в ячейке «Ваш ответ».

**Оценивание.** За каждую практическую работу до 10 баллов: 9–10 – «5», 7–8 – «4», 5–6 – «3».

| ПР №4 | Баллы | ПР №5 | Баллы |
|---|---|---|---|
| шаги 1–2: осмотр и очистка | 3 | шаг 6: аугментация | 3 |
| шаг 3: разбиение и нормализация | 3 | шаги 7–8: поиск гиперпараметров, шум, опасный класс | 3 |
| шаги 4–5: отладка, базовая модель, диагноз | 3 | шаг 9: проверка на тесте | 2 |
| ответы на вопросы и выводы | 1 | шаг 10: отчет и журнал опытов | 2 |

Задания повышенной сложности в конце ноутбука не обязательны; выполненное задание может вернуть один потерянный балл. После сдачи – защита: три вопроса (по вашему ноутбуку, по Блоку 1 и по Блоку 2), она засчитывается как промежуточный контроль по Теме 1.

## Задача

Беспилотный автомобиль должен распознавать дорожные знаки. Детектор уже нашел знак на кадре видеокамеры и вырезал вокруг него квадрат $32 \times 32$ пикселя. Нужно определить, какой это знак. Пока автомобиль подъезжает к знаку, камера снимает его несколько раз: такая серия кадров одного знака называется **проездом**. В проезде знак постепенно растет, а освещение, погода и фон почти не меняются.

| № | Класс | Как выглядит |
|---|---|---|
| 0 | Стоп | красный восьмиугольник, белая кайма, надпись STOP |
| 1 | Уступи дорогу | перевернутый треугольник с красной каймой |
| 2 | Главная дорога | желтый ромб в белой кайме |
| 3 | Пешеходный переход | синий квадрат, белый треугольник, силуэт пешехода |
| 4 | Движение налево | синий круг, белая стрелка с поворотом налево |
| 5 | Движение направо | синий круг, белая стрелка с поворотом направо |
| 6 | Въезд запрещен | красный круг, белая горизонтальная полоса |
| 7 | Ограничение скорости | белый круг с красной каймой, черные цифры |

Самая опасная ошибка: не узнать знак «Стоп». Автомобиль, принявший «Стоп» за другой знак, въедет на перекресток без остановки.

**Что лежит в файле варианта**

| Массив | Что это | Форма и тип |
|---|---|---|
| `X_pool` | кадры для обучения и валидации | $N \times 32 \times 32 \times 3$, `uint8`, яркость 0–255, порядок каналов R, G, B |
| `y_pool` | номер класса кадра | $N$, целые 0–7 |
| `track_pool` | номер проезда, из которого взят кадр | $N$, целые |
| `X_test`, `y_test`, `track_test` | отложенная тестовая выборка из других проездов | то же |
| `classes` | названия классов | 8 строк |

Кадры в массивах перемешаны. Данные выгрузила служба сбора, и про них известно только то, что написано выше. Проверять их – ваша работа.

## Шпаргалка: порядок работы и готовые функции

**Порядок работы над моделью** (Гудфеллоу, глава 11):

1. Определить цель: какую метрику оптимизируем и какое значение нас устроит.
2. Как можно быстрее собрать работающий конвейер «от данных до метрики» с простой стандартной моделью.
3. Поставить диагноз: что мешает больше всего – недообучение, переобучение, ошибки в данных или в коде.
4. Менять по одному: добавлять данные, менять гиперпараметры, регуляризацию или модель по результатам диагноза, записывая каждый опыт.

| Симптом | Диагноз | Что пробовать |
|---|---|---|
| ошибка на обучающей выборке высокая | недообучение | дольше учить, подобрать скорость обучения, взять модель шире или глубже |
| на обучающей низкая, на валидации высокая | переобучение | аугментация, снижение весов, прореживание, больше данных |
| валидация хорошая, а тест намного хуже | утечка или сдвиг данных | проверить разбиение и происхождение данных |
| потери не уменьшаются даже на 64 кадрах | ошибка в коде или данных | проверить метки, нормализацию, форму тензоров, скорость обучения |

**Готовые функции** (определены в готовых ячейках ниже)

| Функция | Что делает |
|---|---|
| `set_seed(seed)` | фиксирует случайность в NumPy и PyTorch |
| `show_frames(X, y=None, n=16, title='')` | рисует до `n` кадров с подписями классов |
| `make_cnn(width=32, bn=True, dropout=0.0)` | сверточная сеть: 5 сверточных слоев, 2 пулинга, глобальное усреднение, выходной слой на 8 классов; `width` – число каналов первого слоя |
| `make_resnet18(n_classes=8)` | ResNet-18 с весами ImageNet из Б2.5 (файл `weights/resnet18-f37072fd.pth`), новой головой на 8 классов и увеличением кадра до $64 \times 64$ внутри модели; нормализация входа – `IMAGENET_NORM` |
| `train_model(model, X_tr, y_tr, X_va, y_va, norm, epochs=15, lr=1e-3, weight_decay=0.0, batch_size=64, augment=None, label_smoothing=0.0, class_weights=None, keep_best=True, seed=SEED, verbose=True)` | обучает AdamW с косинусным расписанием скорости обучения (Б2.1, Б2.3); `norm = (MEAN, STD)` – средние и стандартные отклонения каналов; `augment` – функция, которая получает пакет кадров в диапазоне 0–1 и возвращает измененный пакет; при `keep_best=True` в конце восстанавливает веса лучшей эпохи по F-мере на валидации (ранняя остановка из Б2.3); возвращает историю `hist` |
| `predict(model, X, norm)` | номера классов для кадров `X` (массив NumPy) |
| `scores(y_true, y_pred)` | словарь: точность `acc`, F-мера с макроусреднением `f1`, полнота класса «Стоп» `rec_stop` |
| `plot_history(hist, title='')` | кривые потерь и точности на обучающей и валидационной выборках |
| `plot_confusion(y_true, y_pred, title='')` | матрица ошибок |
| `log(name, hist, **settings)` | записывает опыт в журнал: название, настройки и лучшие метрики на валидации |
| `show_journal()` | журнал опытов в виде таблицы |

**Метрики.** Точность (`acc`) – доля правильных ответов. F-мера класса – среднее гармоническое точности и полноты этого класса: $F_1 = \dfrac{2PR}{P + R}$. F-мера с макроусреднением – среднее F-мер всех восьми классов: редкий класс весит столько же, сколько частый. Полнота класса «Стоп» – доля настоящих знаков «Стоп», которые модель назвала «Стоп».

In [ ]:
# Готовая ячейка 1: библиотеки, вариант, загрузка данных
VARIANT = ...          # <-- номер вашего варианта

import time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, recall_score, confusion_matrix, classification_report

SEED = 42
data = np.load(f'data/road_signs_v{VARIANT:02d}.npz')
CLASSES = [str(c) for c in data['classes']]
STOP = 0                                    # номер класса «Стоп»
X_pool, y_pool, track_pool = data['X_pool'], data['y_pool'], data['track_pool']
X_test, y_test, track_test = data['X_test'], data['y_test'], data['track_test']
print('вариант', VARIANT, '  пул:', X_pool.shape, '  тест:', X_test.shape)


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


def show_frames(X, y=None, n=16, title='', cols=8):
    """Рисует до n кадров; если даны метки y, подписывает класс."""
    n = min(n, len(X))
    rows = (n + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(1.6 * cols, 1.75 * rows))
    for k, ax in enumerate(np.atleast_1d(axs).flat):
        ax.axis('off')
        if k < n:
            ax.imshow(X[k])
            if y is not None:
                ax.set_title(CLASSES[int(y[k])], fontsize=7)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

In [ ]:
# Готовая ячейка 2: сеть, обучение, метрики, рисунки, журнал опытов
def make_cnn(width=32, bn=True, dropout=0.0, n_classes=8):
    """Сеть в духе VGG (Б1.4): [conv-conv-pool] 32 -> 16, [conv-conv-pool] 16 -> 8, conv, глобальное усреднение, Linear."""
    def block(c_in, c_out):
        layers = [nn.Conv2d(c_in, c_out, 3, padding=1, bias=not bn)]
        if bn:
            layers.append(nn.BatchNorm2d(c_out))
        layers.append(nn.ReLU())
        return layers
    w = width
    return nn.Sequential(*block(3, w), *block(w, w), nn.MaxPool2d(2),
                         *block(w, 2 * w), *block(2 * w, 2 * w), nn.MaxPool2d(2),
                         *block(2 * w, 4 * w), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                         nn.Dropout(dropout), nn.Linear(4 * w, n_classes))


IMAGENET_NORM = (np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225]))


def make_resnet18(n_classes=8, weights_path='weights/resnet18-f37072fd.pth', size=64):
    """ResNet-18 с весами ImageNet (Б2.5) и новой головой; кадр увеличивается до size x size внутри модели.
    Вход нормализуется как при предобучении: передавайте в train_model и predict norm=IMAGENET_NORM."""
    import os, torchvision
    net = torchvision.models.resnet18(weights=None)
    if os.path.exists(weights_path):
        net.load_state_dict(torch.load(weights_path, map_location='cpu'))
    else:
        print('файл весов не найден: сеть со случайными весами')
    net.fc = nn.Linear(512, n_classes)
    return nn.Sequential(nn.Upsample(size=(size, size), mode='bilinear', align_corners=False), net)


def to_tensor01(X):
    """uint8 N x 32 x 32 x 3  ->  float N x 3 x 32 x 32 в диапазоне 0–1."""
    return torch.tensor(X).permute(0, 3, 1, 2).float() / 255.0


def _norm_tensors(norm):
    mean, std = norm
    return (torch.tensor(np.asarray(mean), dtype=torch.float32).view(1, 3, 1, 1),
            torch.tensor(np.asarray(std), dtype=torch.float32).view(1, 3, 1, 1))


@torch.no_grad()
def predict_logits(model, X, norm, batch_size=256):
    model.eval()
    mean, std = _norm_tensors(norm)
    out = []
    for k in range(0, len(X), batch_size):
        xb = (to_tensor01(X[k:k + batch_size]) - mean) / std
        out.append(model(xb))
    return torch.cat(out)


def predict(model, X, norm):
    return predict_logits(model, X, norm).argmax(1).numpy()


def scores(y_true, y_pred):
    return {'acc': accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, average='macro', labels=list(range(8)), zero_division=0),
            'rec_stop': recall_score(y_true, y_pred, labels=[STOP], average='macro', zero_division=0)}


def train_model(model, X_tr, y_tr, X_va, y_va, norm, epochs=15, lr=1e-3, weight_decay=0.0, batch_size=64,
                augment=None, label_smoothing=0.0, class_weights=None, keep_best=True, seed=SEED, verbose=True):
    set_seed(seed)
    mean, std = _norm_tensors(norm)
    Xt, yt = to_tensor01(X_tr), torch.tensor(y_tr, dtype=torch.long)
    weight = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=label_smoothing)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc', 'val_f1', 'val_rec_stop']}
    best_f1, best_state, best_epoch = -1.0, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))
        for k in range(0, len(Xt), batch_size):
            idx = perm[k:k + batch_size]
            if len(idx) < 2:                    # пакетной нормализации нужен пакет хотя бы из двух кадров
                continue
            xb = Xt[idx]
            if augment is not None:
                xb = augment(xb)
            loss = loss_fn(model((xb - mean) / std), yt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        # метрики эпохи: на обучающей выборке без аугментации и на валидационной
        lo_tr, lo_va = predict_logits(model, X_tr, norm), predict_logits(model, X_va, norm)
        y_va_t = torch.tensor(y_va, dtype=torch.long)
        s_va = scores(y_va, lo_va.argmax(1).numpy())
        hist['train_loss'].append(F.cross_entropy(lo_tr, yt).item())
        hist['train_acc'].append((lo_tr.argmax(1) == yt).float().mean().item())
        hist['val_loss'].append(F.cross_entropy(lo_va, y_va_t).item())
        hist['val_acc'].append(s_va['acc']); hist['val_f1'].append(s_va['f1']); hist['val_rec_stop'].append(s_va['rec_stop'])
        if s_va['f1'] > best_f1:
            best_f1, best_epoch = s_va['f1'], ep
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"эпоха {ep + 1:2d}: потери {hist['train_loss'][-1]:.3f} / {hist['val_loss'][-1]:.3f}   "
                  f"точность {hist['train_acc'][-1]:.3f} / {hist['val_acc'][-1]:.3f}   F-мера {s_va['f1']:.3f}")
    if keep_best:
        model.load_state_dict(best_state)
    e = best_epoch if keep_best else epochs - 1
    hist['best'] = {'эпоха': e + 1, 'train_acc': hist['train_acc'][e], 'val_acc': hist['val_acc'][e],
                    'val_f1': hist['val_f1'][e], 'val_rec_stop': hist['val_rec_stop'][e]}
    hist['time'] = time.time() - t0
    print(f"готово за {hist['time']:.0f} с; лучшая эпоха {e + 1}: точность на валидации {hist['best']['val_acc']:.3f}, "
          f"F-мера {hist['best']['val_f1']:.3f}, полнота «Стоп» {hist['best']['val_rec_stop']:.3f}")
    return hist


def plot_history(hist, title=''):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ep = range(1, len(hist['train_loss']) + 1)
    ax[0].plot(ep, hist['train_loss'], label='обучающая', color='#0F1B2D'); ax[0].plot(ep, hist['val_loss'], label='валидационная', color='#4C9BFF')
    ax[1].plot(ep, hist['train_acc'], label='обучающая', color='#0F1B2D'); ax[1].plot(ep, hist['val_acc'], label='валидационная', color='#4C9BFF')
    ax[1].plot(ep, hist['val_f1'], label='F-мера, валидация', color='#4C9BFF', ls='--')
    ax[0].set_title('потери'); ax[1].set_title('точность и F-мера')
    for a in ax:
        a.set_xlabel('эпоха'); a.grid(alpha=0.3); a.legend(fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()


def plot_confusion(y_true, y_pred, title=''):
    """Матрица ошибок: строки – правильный класс, столбцы – ответ модели."""
    m = confusion_matrix(y_true, y_pred, labels=list(range(8)))
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    ax.imshow(m, cmap='Blues')
    for i in range(8):
        for j in range(8):
            if m[i, j]:
                ax.text(j, i, m[i, j], ha='center', va='center', fontsize=8, color='white' if m[i, j] > m.max() / 2 else 'black')
    ax.set_xticks(range(8)); ax.set_xticklabels(CLASSES, rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(8)); ax.set_yticklabels(CLASSES, fontsize=8)
    ax.set_xlabel('ответ модели'); ax.set_ylabel('правильный класс'); ax.set_title(title)
    plt.tight_layout(); plt.show()


JOURNAL, MODELS = [], {}


def log(name, hist, model=None, **settings):
    """Записывает опыт в журнал. Модель сохраняется в словарь MODELS под тем же названием."""
    JOURNAL.append({'опыт': name, **settings, **{k: round(v, 3) if isinstance(v, float) else v for k, v in hist['best'].items()},
                    'время, с': round(hist['time'])})
    if model is not None:
        MODELS[name] = model


def show_journal():
    return pd.DataFrame(JOURNAL).fillna('')

---
# ПР №4. Данные, метрика и базовая модель

## Шаг 1. Первый взгляд

1. Посмотрите на формы и типы массивов.
2. Посчитайте для каждого класса число кадров, долю кадров и **число проездов**. Сколько всего проездов и сколько в среднем кадров в одном проезде?
3. Нарисуйте по нескольку кадров каждого класса и все кадры одного проезда.

In [ ]:
# Формы и типы массивов X_pool, X_test
# Для каждого класса: число кадров (np.bincount), доля кадров, число разных проездов (np.unique(track_pool[y_pool == c]))
# Соберите в pd.DataFrame с index=CLASSES. Всего проездов, кадров на проезд в среднем, доля самого частого класса и доля «Стоп»

In [ ]:
# По 8 кадров каждого класса: show_frames(X_pool[индексы], n=8, title=CLASSES[c])
# Все кадры одного проезда (например, проезда track_pool[0]): show_frames(...)

**Вопрос 1.** а) Сколько кадров и сколько проездов в пуле? Сколько в среднем кадров в одном проезде? б) Какая доля у самого частого класса и у класса «Стоп»? Какую точность и какую F-меру с макроусреднением получит модель, которая всегда называет самый частый класс? в) Почему для этой задачи точность – плохая главная метрика и зачем отдельно следить за полнотой класса «Стоп»? г) Чем кадры одного проезда похожи друг на друга? Заметили ли вы что-то странное на рисунках?

*Ваш ответ:*

## Шаг 2. Сбой камеры

Иногда камера выдает испорченный кадр: полностью черный, полностью белый или с полосами через строку. Метку такой кадр получил от проезда, но знака на нем нет.

1. Для каждого кадра пула посчитайте два числа: стандартное отклонение яркости всех его пикселей и среднюю абсолютную разность соседних строк (`np.abs(np.diff(X, axis=1)).mean(...)`). Переведите кадры в `float` перед расчетом.
2. Постройте гистограммы обоих чисел (по оси $y$ удобна логарифмическая шкала) и выберите пороги, отделяющие испорченные кадры от нормальных.
3. Нарисуйте найденные кадры, посчитайте их по видам и удалите их из пула. Проверьте тем же способом тестовую выборку (только проверка, без удаления).

In [ ]:
# 1. Функция frame_stats(X): стандартное отклонение по осям (1, 2, 3) и средняя абсолютная разность соседних строк
# 2. Гистограммы обоих чисел для X_pool, ax.set_yscale('log'); выберите пороги
# 3. bad = (...) | (...); сколько кадров каждого вида (черные, белые, полосатые); show_frames(X_pool[bad], y_pool[bad])
#    X_clean, y_clean, track_clean = пул без испорченных кадров; проверьте тем же способом X_test

**Вопрос 2.** а) Сколько испорченных кадров вы нашли и каких видов? Какие пороги выбрали и почему именно такие? б) Почему испорченные кадры вредны при обучении, хотя их мало? в) Почему пороги мы выбираем по гистограммам, не глядя на метки классов, и почему такую очистку можно делать до разбиения? г) Если сбой камеры случится во время поездки, что должна делать система: доверять ответу сети или чему-то еще?

*Ваш ответ:*

## Шаг 3. Честное разбиение

Валидационная выборка должна быть похожа на то, что модель увидит в работе: **новые проезды**, которых не было при обучении.

1. **Ловушка.** Разбейте очищенный пул случайно по кадрам: 20 % в валидацию. Посчитайте, какая доля валидационных кадров принадлежит проездам, кадры которых есть и в обучающей части.
2. **Честное разбиение по проездам со стратификацией.** Для каждого класса перемешайте его проезды (`np.random.default_rng(SEED)`) и отправьте в валидацию 20 % проездов, но не меньше трех. Все кадры проезда попадают в одну часть.
3. Проверьте: пересечение проездов пустое, в валидации есть все классы. Сколько кадров и проездов в каждой части?
4. Посчитайте средние `MEAN` и стандартные отклонения `STD` каналов R, G, B по **обучающей части** в шкале 0–1 и сохраните `NORM = (MEAN, STD)`.

In [ ]:
# 1. rng = np.random.default_rng(SEED); случайная перестановка кадров, 20 % в валидацию (va_rand, tr_rand)
#    доля валидационных кадров, чей проезд есть в обучении: np.isin(track_clean[va_rand], track_clean[tr_rand]).mean()
# 2. Для каждого класса: проезды класса (np.unique), rng.shuffle, в валидацию max(3, round(0.2 * число проездов))
#    is_val = np.isin(track_clean, val_tracks); X_tr, y_tr, X_va, y_va
# 3. Проверки: пересечение проездов пустое (assert), кадры и проезды по частям, кадры по классам в валидации
# 4. MEAN, STD по осям (0, 1, 2) для X_tr / 255; NORM = (MEAN, STD)

**Вопрос 3.** а) Какая доля валидационных кадров при случайном разбиении «знакома» модели по обучению? Почему это плохо, если все кадры разные? б) Почему разбиение делается по проездам, а не по кадрам? Где еще в задачах беспилотника встречается такая же ловушка? в) Зачем стратификация и почему в валидацию берем не меньше трех проездов каждого класса? г) Почему `MEAN` и `STD` считаются только по обучающей части?

*Ваш ответ:*

## Шаг 4. Метрика, ориентир и отладка

Прежде чем учить большую модель, убедимся, что конвейер работает (Гудфеллоу, раздел 11.5).

1. **Ориентир.** Посчитайте `scores` на валидации для «модели», которая всегда называет самый частый класс обучающей части.
2. **Форма и начальные потери.** Создайте `make_cnn(width=32)`, посчитайте число параметров, пропустите через сеть пакет из 4 кадров (после нормализации) и проверьте форму выхода. Посчитайте потери на этом пакете до обучения и сравните с $\ln 8 \approx 2{,}079$: при случайной инициализации сеть должна быть почти одинаково не уверена во всех классах.
3. **Переобучение крошечной выборки.** Обучите новую сеть на первых 64 кадрах обучающей части 60 эпох (`lr=3e-3`, `keep_best=False`, `verbose=False`), а в качестве валидации передайте те же 64 кадра. Точность на них должна дойти до 1,0. Если не доходит, ищите ошибку в данных или в коде, а не подбирайте гиперпараметры.

In [ ]:
# 1. Самый частый класс обучающей части (np.bincount(y_tr).argmax()); scores(y_va, массив из этого класса)
# 2. set_seed(); model = make_cnn(width=32); число параметров (sum(p.numel() ...))
#    нормализованный пакет из 4 кадров: (to_tensor01(X_tr[:4]) - mean) / std, где mean и std – тензоры формы (1, 3, 1, 1)
#    model.eval(); torch.no_grad(): форма выхода и F.cross_entropy на этом пакете; сравните с np.log(8)
# 3. h_tiny = train_model(make_cnn(width=32), первые 64 кадра X_tr и y_tr, они же как валидация, NORM,
#                         epochs=60, lr=3e-3, keep_best=False, verbose=False); точность h_tiny['train_acc'][-1]

**Вопрос 4.** а) Какие точность и F-меру дает ориентир? Почему F-мера у него такая маленькая? б) Сколько параметров у сети и какая форма выхода? Близки ли начальные потери к $\ln 8$? Что значило бы, если бы они были, например, 15? в) Дошла ли точность на 64 кадрах до 1,0? Почему эта проверка полезна, хотя модель, запомнившая 64 кадра, нам не нужна?

*Ваш ответ:*

## Шаг 5. Базовая модель и диагноз

1. Обучите базовую модель: `make_cnn(width=32)`, 15 эпох, остальные параметры по умолчанию, без аугментации. Запишите опыт в журнал: `log('базовая', h_base, model, разбиение='по проездам', аугментация='нет')`. Нарисуйте кривые и матрицу ошибок на валидации.
2. **Цена ловушки.** Обучите такую же модель на случайном разбиении по кадрам из шага 3 (`tr_rand`, `va_rand`; нормализацию оставьте `NORM`) и запишите в журнал с пометкой `разбиение='по кадрам'`.
3. Сравните валидационные оценки двух опытов и поставьте диагноз базовой модели по таблице из шпаргалки.

In [ ]:
# 1. set_seed(); model_base = make_cnn(width=32); h_base = train_model(..., NORM, epochs=15)
#    log('базовая', h_base, model_base, разбиение='по проездам', аугментация='нет')
#    plot_history(h_base); plot_confusion(y_va, predict(model_base, X_va, NORM))
# 2. Та же модель на X_clean[tr_rand], y_clean[tr_rand] с валидацией X_clean[va_rand], y_clean[va_rand]
#    log('базовая, ловушка', h_leak, разбиение='по кадрам', аугментация='нет'); show_journal()

**Вопрос 5.** а) Какие точность на обучающей выборке и F-мера на валидации у базовой модели? Какой диагноз вы ставите и по каким числам? б) Насколько оценка при разбиении по кадрам выше честной? Какую из двух оценок вы бы показали заказчику и почему? в) Какие пары классов модель путает чаще всего? Почему именно эти? г) Что вы будете пробовать в первую очередь в ПР №5 и почему?

*Ваш ответ:*

### Выводы по ПР №4

*Заполните своими словами и своими числами.*

- Данные: сколько кадров, проездов, классов; доля «Стоп».
- Проблемы и исправления: что найдено в данных и что с этим сделано.
- Разбиение: как и почему; сколько кадров и проездов в частях.
- Отладка: начальные потери, переобучение 64 кадров.
- Базовая модель и диагноз: числа и вывод.

---
# ПР №5. Улучшение модели и итоговая проверка

**Бюджет:** не больше 8 обучений в шагах 6–8. Меняйте по одному и записывайте каждый опыт в журнал. Обучение по 15–20 эпох на процессоре занимает около минуты; если компьютер медленный, уменьшите `epochs` во всех опытах одинаково.

## Шаг 6. Аугментация

Напишите функцию `augment(x)`: на вход пакет `x` формы $B \times 3 \times 32 \times 32$ в диапазоне 0–1, на выход пакет той же формы. Каждому кадру – свое случайное преобразование (Б2.4):

1. **сдвиг** на целое число пикселей от −3 до 3 по каждой оси: дополнить кадр на 3 пикселя повтором краев (`F.pad(x, (3, 3, 3, 3), mode='replicate')`) и вырезать окно $32 \times 32$ со случайным началом;
2. **яркость и контраст**: $x \leftarrow (x - \bar x) \cdot c + \bar x$, затем $x \leftarrow x \cdot b$, где $\bar x$ – средняя яркость кадра, $c$ и $b$ случайны от 0,7 до 1,3 и от 0,6 до 1,4; результат обрезать до диапазона 0–1.

Затем напишите `augment_flip(x)`: то же самое плюс горизонтальное отражение половины кадров (`torch.flip(x, dims=[3])`).

Проведите два опыта по 20 эпох: с `augment` и с `augment_flip`. Сравните журнал и матрицы ошибок.

In [ ]:
# def augment(x): сдвиг (F.pad с mode='replicate' и случайное окно 32 x 32 для каждого кадра),
#                 контраст c из [0.7, 1.3] и яркость b из [0.6, 1.4] для каждого кадра, clamp(0, 1)
# def augment_flip(x): augment и torch.flip(..., dims=[3]) для случайной половины кадров
# Покажите 16 кадров после augment: show_frames(augment(to_tensor01(X_tr[:16])).permute(0, 2, 3, 1).numpy(), y_tr[:16])
# Два опыта по 20 эпох (augment=augment и augment=augment_flip), каждый в журнал; матрицы ошибок; show_journal()

**Вопрос 6.** а) Как изменились точность на обучающей выборке и F-мера на валидации после аугментации? Как изменился разрыв между ними? б) Что сделало отражение с классами «Движение налево» и «Движение направо»? Почему? Какие еще знаки отражение не портит? в) Какие преобразования еще подошли бы для этой задачи, а какие нельзя применять (кроме отражения)?

*Ваш ответ:*

## Шаг 7. Случайный поиск гиперпараметров

Гудфеллоу (раздел 11.4) советует при нескольких гиперпараметрах не перебирать сетку, а брать **случайные** сочетания: при том же числе опытов каждый важный гиперпараметр принимает больше разных значений. Скорость обучения и коэффициент снижения весов выбирают в **логарифмической шкале**.

1. С аугментацией `augment` проведите 3 опыта по 20 эпох со случайными сочетаниями (генератор `np.random.default_rng(SEED)`):
   - `lr` = $10^u$, где $u$ равномерно от −3,5 до −2,3;
   - `weight_decay` – из списка 0, 1e-4, 1e-3, 1e-2;
   - `dropout` – из списка 0, 0,3;
   - `width` – из списка 32, 48.
2. Лучший по F-мере на валидации опыт повторите с другим зерном (`seed=7`), чтобы оценить, насколько результат зависит от случайности.
3. **По желанию, в пределах бюджета:** если рядом с ноутбуком есть папка `weights` с весами ResNet-18 из Б2.5, сделайте опыт с предобученной сетью: `make_resnet18()`, нормализация `IMAGENET_NORM`, 8 эпох, `lr=3e-4`, `augment=augment`. Одна эпоха на процессоре занимает в несколько раз больше времени, чем у `make_cnn`.

In [ ]:
# rng_hp = np.random.default_rng(SEED); три опыта: lr = 10 ** rng_hp.uniform(-3.5, -2.3), weight_decay, dropout, width – rng_hp.choice
# Каждый опыт: set_seed(); make_cnn(width=..., dropout=...); train_model(..., epochs=20, lr=..., weight_decay=..., augment=augment)
#              log(f'поиск {k + 1}', h, m, аугментация='сдвиг, яркость', эпох=20, lr=..., weight_decay=..., dropout=..., width=...)
# Лучший по val_f1 повторите с seed=7 (и set_seed(7) перед make_cnn), запишите в журнал
# По желанию: make_resnet18(), train_model(..., IMAGENET_NORM, epochs=8, lr=3e-4, augment=augment), log(...); show_journal()

**Вопрос 7.** а) Какое сочетание оказалось лучшим? Насколько оно лучше опыта «аугментация» из шага 6? б) Насколько изменилась F-мера при смене зерна? Можно ли после этого утверждать, что лучшее сочетание лучше остальных? в) Почему скорость обучения выбирают в логарифмической шкале? Почему случайный поиск при том же числе опытов обычно находит лучшие значения, чем сетка?

*Ваш ответ:*

## Шаг 8. Опасный класс

Посмотрите в журнале на полноту класса «Стоп» у лучших опытов. Чтобы модель реже пропускала «Стоп», можно увеличить вес ошибки на этом классе в функции потерь: `class_weights` – список из 8 чисел, у класса «Стоп» вес 3, у остальных 1.

1. Повторите лучшее сочетание с `class_weights` и запишите в журнал.
2. Сравните F-меру и полноту «Стоп» с тем же сочетанием без весов. Выберите итоговую модель и обоснуйте выбор: какой метрике вы отдаете приоритет и почему.

In [ ]:
# w = [3.0] + [1.0] * 7; то же лучшее сочетание с class_weights=w; log(..., вес_стоп=3); show_journal()

**Вопрос 8.** а) Как изменились полнота «Стоп» и F-мера после увеличения веса? Чем пришлось заплатить? б) Какую модель вы выбираете итоговой и почему? в) Какое значение полноты «Стоп» вы сочли бы достаточным для настоящего автомобиля? Можно ли добиться его одним классификатором кадров?

*Ваш ответ:*

## Шаг 9. Итоговая проверка на тестовой выборке

Модель выбрана, теперь **один раз** проверьте ее на тестовой выборке. Модели сохранены в словаре `MODELS` под названиями из журнала.

1. Точность, F-мера с макроусреднением, полнота «Стоп» на тесте; отчет по классам (`classification_report`) и матрица ошибок.
2. Число самых опасных ошибок: кадры «Стоп», которым модель дала другой класс.
3. **Решение по проезду.** Автомобиль видит знак на нескольких кадрах. Для каждого проезда тестовой выборки возьмите самый частый ответ модели на его кадрах (голосование) и посчитайте долю проездов, распознанных правильно, и число пропущенных проездов «Стоп».

In [ ]:
# FINAL = название итоговой модели из журнала; final_model = MODELS[FINAL]; y_pred = predict(final_model, X_test, NORM)
#   (если итоговая модель – ResNet-18, вместо NORM нужна IMAGENET_NORM)
# scores(y_test, y_pred); classification_report(..., target_names=CLASSES, digits=3); plot_confusion(...)
# Опасные ошибки: (y_test == STOP) & (y_pred != STOP)
# Голосование: pd.DataFrame({'проезд': track_test, 'класс': y_test, 'ответ': y_pred}).groupby('проезд')
#              .agg(класс=('класс', 'first'), ответ=('ответ', lambda s: s.mode().iloc[0]))
#              доля проездов с правильным ответом; число проездов «Стоп» с неверным ответом

**Вопрос 9.** а) Совпала ли оценка на тесте с оценкой на валидации? Какое расхождение можно считать нормальным, если в валидации всего около 45 проездов? б) Какие классы путаются на тесте? Совпадает ли это с валидацией? в) Насколько голосование по проезду лучше решения по одному кадру? Почему оно помогает? г) Почему после этого шага нельзя вернуться и «еще немного подкрутить» модель, глядя на тест?

*Ваш ответ:*

## Шаг 10. Отчет

Заполните отчет. Пишите так, чтобы его понял человек, который не видел ваш код: что за задача, что сделано, какие числа получены и что из них следует. Журнал опытов – обязательная часть отчета.

In [ ]:
show_journal()

### Отчет

**Задача.** *Что распознаем, по каким данным, какая главная метрика и почему, какая ошибка самая опасная.*

**Данные.** *Сколько кадров и проездов, какие проблемы найдены и как исправлены, как сделано разбиение и почему.*

**Отладка и базовая модель.** *Начальные потери, проверка на 64 кадрах, результаты базовой модели, диагноз, цена ловушки разбиения.*

**Опыты.** *Что пробовали и в каком порядке (по журналу), что помогло, что нет, насколько велик шум от зерна.*

**Итоговая проверка.** *Какая модель выбрана и почему; точность, F-мера и полнота «Стоп» на тесте; сравнение с валидацией; голосование по проезду; какие классы путаются.*

**Риски и что улучшить.** *Что может пойти не так на настоящей дороге и что бы вы сделали дальше.*

---
### Задания повышенной сложности

1. **Предобученная сеть.** Возьмите ResNet-18 с весами ImageNet из Б2.5 (файл `weights/resnet18-f37072fd.pth`), увеличьте кадры до $64 \times 64$, нормализуйте как при предобучении и дообучите с разными скоростями для основы и головы. Помогает ли перенос на этих данных? Сколько стоит одна эпоха по времени?
2. **Аугментация при проверке.** Для итоговой модели усредните вероятности по исходному кадру и 4 сдвинутым на 2 пикселя копиям (Б2.4). Как изменились F-мера и полнота «Стоп» на валидации?
3. **Отказ от ответа.** Если максимальная вероятность меньше порога, модель отвечает «не уверена». Постройте на валидации зависимость доли отказов и точности на оставшихся кадрах от порога. Где разумно поставить порог для знака «Стоп»?
4. **Нужны ли новые данные.** Обучите лучшее сочетание на 25 %, 50 % и 100 % обучающих проездов и постройте F-меру на валидации от числа проездов (Гудфеллоу, раздел 11.3). Что выгоднее: собирать новые проезды или подбирать модель?
5. **Что видит сеть.** Для нескольких кадров «Стоп», принятых за «Въезд запрещен», закрывайте серым квадратом $6 \times 6$ разные места кадра и смотрите, как меняется вероятность класса «Стоп» (карта перекрытия). На какие пиксели опирается решение?

In [ ]:
# Место для заданий повышенной сложности